# Train Model A - Burnout Classifier

The original Model A (`modelA.pkl`) was 201 MB (XGBoost, 2019 estimators, depth 12, trained on 2.1 million rows with SMOTE); too large to transpile to JavaScript via m2cgen for web deployment.

This notebook retrains Model A with a much smaller configuration:
- **Same algorithm**: XGBoost (supported by m2cgen)
- **n_estimators**: 2019 → 200
- **max_depth**: 12 → 5
- **Same preprocessing**: IQR cleaning, `engineer_features_a`, SMOTE

Output: `models/modelA.pkl` (~1.6 MB, Macro F1 ~0.53; on par with the original)

## 1. Imports & Config

In [10]:
import sys
import os
import time
import numpy as np
import pandas as pd
import joblib
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.features import engineer_features_a

In [11]:
DATA_PATH = os.path.join(REPO_ROOT, "datasets", "academic_stress_level.csv")
OUT_PATH  = os.path.join(REPO_ROOT, "models", "modelA.pkl")

FEATURE_COLS = [
    "study_hours_per_day", "sleep_hours", "exam_pressure", "stress_level",
    "financial_stress", "social_support", "anxiety_score", "depression_score",
    "family_expectation", "physical_activity",
]
CLASS_NAMES = {0: "Healthy", 1: "Mildly Burnout", 2: "Burnout"}

N_ESTIMATORS  = 200
MAX_DEPTH     = 5
LEARNING_RATE = 0.15

## 2. Load & Preprocess Data

In [12]:
t0 = time.time()
print(f"Loading {DATA_PATH} ...")
df = pd.read_csv(DATA_PATH)
print(f"Raw dataset: {len(df):,} rows")

def bin_burnout(score):
    if score < 4:   return 0
    elif score < 7: return 1
    return 2

df["burnout_class"] = df["burnout_score"].apply(bin_burnout)
df_model = df[FEATURE_COLS + ["burnout_class"]].dropna()

# IQR cleaning on features only (preserves all class samples)
for col in FEATURE_COLS:
    q1, q3 = df_model[col].quantile(0.25), df_model[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    mask = (df_model[col] >= q1 - 1.5 * iqr) & (df_model[col] <= q3 + 1.5 * iqr)
    df_model = df_model[mask]

print(f"After IQR clean: {len(df_model):,} rows")

Loading c:\Users\asus\Downloads\AcademicShield\datasets\academic_stress_level.csv ...
Raw dataset: 1,000,000 rows
After IQR clean: 981,357 rows


## 3. Feature Engineering

In [13]:
df_model = engineer_features_a(df_model)
all_features = [c for c in df_model.columns if c != "burnout_class"]
print(f"Total features after engineering: {len(all_features)}")

X = df_model[all_features]
y = df_model["burnout_class"]

Total features after engineering: 20


## 4. Train / Test Split + SMOTE

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Applying SMOTE to training data ...")
X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f"After SMOTE: {len(X_train_sm):,} training rows")

Applying SMOTE to training data ...
After SMOTE: 2,105,241 training rows


## 5. Training

In [15]:
print(f"Training compact XGBClassifier (n_estimators={N_ESTIMATORS}, max_depth={MAX_DEPTH}) ...")
clf = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
clf.fit(X_train_sm, y_train_sm)
clf.feature_names_ = list(all_features)

Training compact XGBClassifier (n_estimators=200, max_depth=5) ...


## 6. Evaluation

In [16]:
y_pred = clf.predict(X_test)
acc  = accuracy_score(y_test, y_pred)
f1m  = f1_score(y_test, y_pred, average="macro")

print("=== MODEL A — TEST METRICS ===")
print(f"Accuracy : {acc:.4f}")
print(f"Macro-F1 : {f1m:.4f}")
print(classification_report(y_test, y_pred, target_names=list(CLASS_NAMES.values())))

=== MODEL A — TEST METRICS ===
Accuracy : 0.8465
Macro-F1 : 0.5279
                precision    recall  f1-score   support

       Healthy       0.98      0.86      0.92    175437
Mildly Burnout       0.38      0.73      0.50     20405
       Burnout       0.09      0.71      0.17       430

      accuracy                           0.85    196272
     macro avg       0.49      0.77      0.53    196272
  weighted avg       0.92      0.85      0.87    196272



## 6b. Cross-Validation

5-fold StratifiedKFold with SMOTE inside each fold (no leakage), confirms the compact model is stable, not overfit to one split.

In [17]:
import time
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline

def _build_clf():
    return xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, learning_rate=LEARNING_RATE,
        objective="multi:softprob", num_class=3, tree_method="hist",
        random_state=42, n_jobs=-1, verbosity=0,
    )

tcv = time.time()
pipe = ImbPipeline([("smote", SMOTE(random_state=42)), ("clf", _build_clf())])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(pipe, X, y, cv=skf, scoring=["accuracy", "f1_macro"], n_jobs=1)

print("=== COMPACT MODEL A — CROSS-VALIDATED (5-fold) ===")
print(f"CV Accuracy : {cv['test_accuracy'].mean():.4f} +/- {cv['test_accuracy'].std():.4f}")
print(f"CV Macro-F1 : {cv['test_f1_macro'].mean():.4f} +/- {cv['test_f1_macro'].std():.4f}")
print(f"(CV took {time.time() - tcv:.1f}s)")


=== COMPACT MODEL A — CROSS-VALIDATED (5-fold) ===
CV Accuracy : 0.8464 +/- 0.0006
CV Macro-F1 : 0.5265 +/- 0.0021
(CV took 228.4s)


## 7. Save Model

In [18]:
joblib.dump(clf, OUT_PATH)
size_mb  = os.path.getsize(OUT_PATH) / 1e6
n_trees  = len(clf.get_booster().get_dump())
print(f"Saved {OUT_PATH}  ({size_mb:.3f} MB, {n_trees} trees)")
print(f"Feature order: {all_features}")
print(f"Done in {time.time() - t0:.1f}s")

Saved c:\Users\asus\Downloads\AcademicShield\models\modelA.pkl  (1.606 MB, 600 trees)
Feature order: ['study_hours_per_day', 'sleep_hours', 'exam_pressure', 'stress_level', 'financial_stress', 'social_support', 'anxiety_score', 'depression_score', 'family_expectation', 'physical_activity', 'stress_x_anxiety', 'stress_x_depression', 'stress_x_exam', 'anxiety_x_depression', 'study_sleep_ratio', 'pressure_support_gap', 'stress_level_sq', 'anxiety_score_sq', 'sleep_deprived', 'high_stress']
Done in 281.0s
